In [8]:
%pip install yfinance pandas numpy matplotlib seaborn

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Notebook 01 — Data Ingestion

Pulls daily prices and ESG sustainability scores for a seed universe of US-listed equities via yfinance.

**Universe:** 50 large-cap US stocks, ~10 per sector (tech, financials, energy/materials, healthcare, consumer).
**Date range:** 2018-01-01 to 2024-12-31.
**Outputs:** `data/raw/prices.csv`, `data/raw/sustainability_scores.csv`.

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

print("pandas:", pd.__version__)
print("yfinance:", yf.__version__)

pandas: 2.2.2
yfinance: 1.3.0


In [2]:
# Seed universe — 50 US-listed large-caps, roughly balanced across sectors
SEED_UNIVERSE = {
    "tech":        ["AAPL", "MSFT", "GOOGL", "META", "NVDA", "AMZN", "ORCL", "CRM", "ADBE", "INTC"],
    "financials":  ["JPM", "BAC", "WFC", "GS", "MS", "C", "AXP", "BLK", "SPGI", "MSCI"],
    "energy_mat":  ["XOM", "CVX", "COP", "SLB", "OXY", "LIN", "APD", "ECL", "DOW", "FCX"],
    "healthcare":  ["JNJ", "PFE", "UNH", "ABBV", "MRK", "LLY", "TMO", "ABT", "CVS", "BMY"],
    "consumer":    ["WMT", "KO", "PEP", "PG", "MCD", "NKE", "SBUX", "HD", "COST", "TGT"],
}

TICKERS = [t for sector in SEED_UNIVERSE.values() for t in sector]
START, END = "2018-01-01", "2024-12-31"

RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Universe size: {len(TICKERS)} tickers")
print(f"Date range: {START} to {END}")
print(f"Writing to: {RAW_DIR.resolve()}")

Universe size: 50 tickers
Date range: 2018-01-01 to 2024-12-31
Writing to: /Users/jimmy/Documents/esg-retail-flows-causal/data/raw


In [3]:
# Bulk download all tickers in one call — yfinance handles batching
raw = yf.download(
    tickers=TICKERS,
    start=START,
    end=END,
    auto_adjust=False,
    progress=True,
    group_by="ticker",
)

# Reshape into long format: one row per (ticker, date)
frames = []
for ticker in TICKERS:
    if ticker not in raw.columns.get_level_values(0):
        print(f"  skipped {ticker} — no data returned")
        continue
    df = raw[ticker].copy()
    df["ticker"] = ticker
    df = df.reset_index()
    frames.append(df)

prices = pd.concat(frames, ignore_index=True)
prices.columns = [c.lower().replace(" ", "_") for c in prices.columns]
prices = prices[["ticker", "date", "open", "high", "low", "close", "adj_close", "volume"]]

print(f"\nRows: {len(prices):,}")
print(f"Tickers with data: {prices['ticker'].nunique()} / {len(TICKERS)}")
print(f"Date span: {prices['date'].min().date()} to {prices['date'].max().date()}")
prices.head()

[*********************100%***********************]  50 of 50 completed


Rows: 88,000
Tickers with data: 50 / 50
Date span: 2018-01-02 to 2024-12-30


,ticker,date,open,high,low,close,adj_close,volume
0,AAPL,2018-01-02,42.540001,43.075001,42.314999,43.064999,40.304184,102223600.0
1,AAPL,2018-01-03,43.132500,43.637501,42.990002,43.057499,40.297146,118071600.0
2,AAPL,2018-01-04,43.134998,43.367500,43.020000,43.257500,40.484329,89738400.0
3,AAPL,2018-01-05,43.360001,43.842499,43.262501,43.750000,40.945263,94640000.0
4,AAPL,2018-01-08,43.587502,43.902500,43.482498,43.587502,40.793179,82271200.0


In [4]:
# Save prices to raw data folder
prices_path = RAW_DIR / "prices.csv"
prices.to_csv(prices_path, index=False)
print(f"Saved {len(prices):,} rows to {prices_path}")
print(f"File size: {prices_path.stat().st_size / 1024 / 1024:.1f} MB")

Saved 88,000 rows to ../data/raw/prices.csv
File size: 9.5 MB


In [7]:
# Yahoo Finance sustainability endpoint is currently returning 404 for most tickers
# (Yahoo deprecated their ESG scoring feed around 2022-2023).
# We attempt the pull but don't treat failure as a blocker — ESG-INDEX MEMBERSHIP
# (MSCI ESG Leaders, S&P 500 ESG) is the load-bearing treatment variable, not Yahoo's score.
# We'll pull index membership in Notebook 02.

import io, contextlib

sustainability_rows = []
missing = []

# Suppress yfinance's noisy per-ticker error printing
buf = io.StringIO()
with contextlib.redirect_stderr(buf), contextlib.redirect_stdout(buf):
    for ticker in TICKERS:
        try:
            t = yf.Ticker(ticker)
            sust = t.sustainability
            if sust is None or (hasattr(sust, "empty") and sust.empty):
                missing.append(ticker)
                continue
            row = sust.to_dict().get("esgScores", {})
            row["ticker"] = ticker
            sustainability_rows.append(row)
        except Exception:
            missing.append(ticker)

sustainability = pd.DataFrame(sustainability_rows)
print(f"Yahoo sustainability endpoint coverage: {len(sustainability)} / {len(TICKERS)} tickers")
print(f"Failures: {len(missing)} (Yahoo API returning 404 for ESG data — expected).")
print("NOTE: This is a known data limitation. ESG-index inclusion membership will be")
print("pulled in Notebook 02 and serves as the primary treatment variable.")

Yahoo sustainability endpoint coverage: 0 / 50 tickers
Failures: 50 (Yahoo API returning 404 for ESG data — expected).
NOTE: This is a known data limitation. ESG-index inclusion membership will be
pulled in Notebook 02 and serves as the primary treatment variable.


In [8]:
# Save an empty sustainability file to document the coverage gap explicitly
sustainability_path = RAW_DIR / "sustainability_scores.csv"
if len(sustainability) > 0:
    sustainability.to_csv(sustainability_path, index=False)
    print(f"Saved {len(sustainability)} rows to {sustainability_path}")
else:
    # Write a placeholder so downstream code can check file existence
    pd.DataFrame(columns=["ticker", "note"]).to_csv(sustainability_path, index=False)
    print(f"Wrote empty placeholder to {sustainability_path}")
    print("Yahoo ESG endpoint is returning 404 for all tickers — documented.")

Wrote empty placeholder to ../data/raw/sustainability_scores.csv
Yahoo ESG endpoint is returning 404 for all tickers — documented.


In [ ]:
# Final checks before closing the notebook
print("=" * 60)
print("NOTEBOOK 01 PART 1 — VALIDATION SUMMARY")
print("=" * 60)

print("\nPRICES")
print(f"  Rows:               {len(prices):,}")
print(f"  Tickers with data:  {prices['ticker'].nunique()} / {len(TICKERS)}")
print(f"  Date range:         {prices['date'].min().date()} to {prices['date'].max().date()}")
print(f"  Trading days/ticker (median): {prices.groupby('ticker').size().median():.0f}")
print(f"  NaN counts per col:\n{prices.isna().sum().to_string()}")

print("\nSUSTAINABILITY")
print(f"  Tickers with ESG data:  {len(sustainability)} / {len(TICKERS)}")
print(f"  ESG data coverage:      {100 * len(sustainability) / len(TICKERS):.0f}%")
if missing:
    print(f"  Missing tickers:        {', '.join(missing)}")

print("\nOUTPUTS")
print(f"  {prices_path}")
print(f"  {sustainability_path}")